# Punto de Control: Análisis de decisiones de limpieza
**Objetivo:** Practicar decisiones reales sobre cómo tratar problemas en datos.

Basado en dos casos reales:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 20)
print('✓ Librerías cargadas')

## Cargar datos reales

In [ ]:
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_estacion.csv'
df = pd.read_csv(url)
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'], format='mixed', errors='coerce')
print(f'Dataset: {df.shape}')
print(df[['fecha_hora', 'irradiancia_wm2', 'velocidad_viento_kmh']].head(10))

---
# CASO A: IRRADIANCIA SOLAR
**Problema:** Un sensor de irradiancia tiene 3% de datos faltantes, distribuidos en horas aleatorias del día.

**Pregunta:** ¿Eliminar, rellenar con la mediana o interpolar? ¿Por qué?

## Paso 1: Simular el problema

In [ ]:
df_irr = df.copy()
rng = np.random.default_rng(42)

# Crear 3% de faltantes
indices_nan = rng.choice(len(df_irr), int(len(df_irr)*0.03), replace=False)
df_irr.loc[indices_nan, 'irradiancia_wm2'] = np.nan

print(f'Faltantes introducidos: {df_irr["irradiancia_wm2"].isna().sum()} ({df_irr["irradiancia_wm2"].isna().sum()/len(df_irr)*100:.1f}%)')
print(f'Distribuidos en horas:')
print(df_irr[df_irr['irradiancia_wm2'].isna()]['fecha_hora'].dt.hour.value_counts().sort_index())

## Paso 2: Visualizar el problema

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# Línea completa
ax.plot(df_irr['irradiancia_wm2'].values, color='blue', alpha=0.6, linewidth=1, label='Con huecos')

# Marcar faltantes
ax.scatter(df_irr[df_irr['irradiancia_wm2'].isna()].index, 
          [0]*df_irr['irradiancia_wm2'].isna().sum(),
          color='red', s=50, marker='X', label='Faltantes (NaN)', zorder=5)

ax.set_title('CASO A: Irradiancia con 3% de faltantes aleatorios', fontweight='bold', fontsize=12)
ax.set_ylabel('Irradiancia (W/m²)')
ax.set_xlabel('Tiempo')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Observación: Los huecos rompen la continuidad de la curva solar')

## Paso 3: Comparar 3 estrategias

In [ ]:
# Estrategia 1: ELIMINAR
s1_eliminar = df_irr['irradiancia_wm2'].dropna()

# Estrategia 2: MEDIANA
s2_mediana = df_irr['irradiancia_wm2'].fillna(df_irr['irradiancia_wm2'].median())

# Estrategia 3: INTERPOLAR (recomendada)
df_irr_sorted = df_irr.sort_values('fecha_hora').reset_index(drop=True)
s3_interpolar = df_irr_sorted['irradiancia_wm2'].interpolate(method='linear')

print('COMPARACIÓN DE ESTRATEGIAS:')
print(f'\n1. ELIMINAR:')
print(f'   - Datos restantes: {len(s1_eliminar)} (perdemos {len(df_irr)-len(s1_eliminar)})')
print(f'   - Problema: Rompe continuidad temporal')

print(f'\n2. MEDIANA (valor: {df_irr["irradiancia_wm2"].median():.0f}):')
print(f'   - Todos los datos: {len(s2_mediana)}')
print(f'   - Problema: Crea picos artificiales (valor plano)')

print(f'\n3. INTERPOLAR (RECOMENDADO):')
print(f'   - Todos los datos: {len(s3_interpolar)}')
print(f'   - Ventaja: Mantiene continuidad y variación natural')

## Paso 4: Visualizar comparación

In [ ]:
# Seleccionar rango pequeño para ver claridad
rango = slice(100, 150)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Estrategia 1: Eliminar
ax = axes[0]
s1_rango = s1_eliminar[rango]
ax.scatter(s1_rango.index, s1_rango.values, color='red', s=50, edgecolors='darkred', linewidth=1.5)
ax.set_title('1. ELIMINAR (pierdes datos y continuidad)', fontweight='bold', color='darkred')
ax.set_ylabel('W/m²')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1000)

# Estrategia 2: Mediana
ax = axes[1]
s2_rango = s2_mediana[rango]
ax.plot(s2_rango.index, s2_rango.values, color='orange', marker='o', markersize=4, linewidth=1)
ax.axhline(df_irr['irradiancia_wm2'].median(), color='orange', linestyle='--', alpha=0.5, label=f'Mediana={df_irr["irradiancia_wm2"].median():.0f}')
ax.set_title('2. MEDIANA (crea picos artificiales)', fontweight='bold', color='darkorange')
ax.set_ylabel('W/m²')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1000)

# Estrategia 3: Interpolar
ax = axes[2]
s3_rango = s3_interpolar[rango]
ax.plot(s3_rango.index, s3_rango.values, color='green', marker='^', markersize=4, linewidth=2, label='Interpolado')
ax.scatter(df_irr_sorted[rango][df_irr_sorted[rango]['irradiancia_wm2'].isna()].index,
          s3_rango[df_irr_sorted[rango]['irradiancia_wm2'].isna()].values,
          color='darkgreen', s=100, marker='D', edgecolors='black', linewidth=2, label='Rellenados', zorder=5)
ax.set_title('3. INTERPOLAR (MEJOR) ✓ Mantiene forma natural', fontweight='bold', color='darkgreen')
ax.set_ylabel('W/m²')
ax.set_xlabel('Índice')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1000)

plt.tight_layout()
plt.show()

## CONCLUSIÓN CASO A

✅ **RESPUESTA: INTERPOLAR**

**¿Por qué?**
1. Es **serie de tiempo** → valores vecinos son representativos
2. Irradiancia **varía suavemente** hora a hora
3. Mantiene **forma natural** de la curva solar
4. Pierdes **poco** (3%) vs. perder continuidad

**Código:**
```python
df['irradiancia_wm2'] = df['irradiancia_wm2'].sort_values('fecha_hora').interpolate(method='linear')
```

---
# CASO B: VIENTO EXTREMO
**Problema:** Una columna de velocidad de viento tiene valores de 400 m/s en 12 de 50,000 registros.

**Pregunta:** ¿Es error o evento real? ¿Cómo lo confirmarían?

## Paso 1: Simular el problema

In [ ]:
df_viento = df.copy()

# Introducir 12 valores de 400 m/s
indices_extremo = rng.choice(len(df_viento), 12, replace=False)
df_viento.loc[indices_extremo, 'velocidad_viento_kmh'] *= 100  # Simula ×100 (error de sensor)

print(f'Valores extremos introducidos:')
print(df_viento[df_viento['velocidad_viento_kmh'] > 100][['fecha_hora', 'velocidad_viento_kmh']].head())
print(f'\nTotal de extremos: {(df_viento["velocidad_viento_kmh"] > 100).sum()} de {len(df_viento)} ({(df_viento["velocidad_viento_kmh"] > 100).sum()/len(df_viento)*100:.3f}%)')

## Paso 2: Análisis estadístico (IQR)

In [ ]:
col = 'velocidad_viento_kmh'

# Calcular IQR
q1 = df_viento[col].quantile(0.25)
q3 = df_viento[col].quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

print('ANÁLISIS CON REGLA IQR:')
print(f'\nQ1 (25%): {q1:.2f} km/h')
print(f'Q3 (75%): {q3:.2f} km/h')
print(f'IQR: {iqr:.2f} km/h')
print(f'\nLímites normales:')
print(f'  Inferior: {lim_inf:.2f} km/h')
print(f'  Superior: {lim_sup:.2f} km/h')
print(f'\nVelocidades encontradas:')
print(f'  Máxima normal: {df_viento[df_viento[col] <= lim_sup][col].max():.2f} km/h')
print(f'  Máxima extrema: {df_viento[col].max():.2f} km/h')
print(f'\nAtípicos detectados: {(df_viento[col] > lim_sup).sum()}')

## Paso 3: Verificar contexto

In [ ]:
extremos = df_viento[df_viento['velocidad_viento_kmh'] > 100]

print('ANÁLISIS DE CONTEXTO:')
print(f'\nVálores extremos encontrados:')
print(extremos[['fecha_hora', 'velocidad_viento_kmh', 'temperatura_c', 'presion_hpa', 'irradiancia_wm2']])

print(f'\n¿Hay otros datos anómalos en esas filas?')
print(f'  - Temperatura extrema: {extremos["temperatura_c"].std():.2f}°C (normal: {df_viento["temperatura_c"].std():.2f}°C)')
print(f'  - Presión extrema: {extremos["presion_hpa"].std():.2f} hPa (normal: {df_viento["presion_hpa"].std():.2f} hPa)')
print(f'\n¿Pasaron en el mismo sensor o todos?')
print(f'  - Solo en velocidad_viento_kmh (otras columnas normales) → Posible error del sensor')

## Paso 4: Decisión y acción

In [ ]:
print('DIAGNÓSTICO:')
print('\n1. Magnitud imposible: 400 m/s = 1440 km/h')
print('   → Viento más fuerte registrado: ~408 km/h')
print('\n2. Frecuencia extremadamente rara: 12/50000 = 0.024%')
print('   → Normal sería ~0.3-1% de atípicos')
print('\n3. Aislados en una columna')
print('   → Temperatura/presión normales en esas fechas')
print('   → Sugiere error del sensor, no evento meteorológico')
print('\n✅ CONCLUSIÓN: ES ERROR (no evento real)')
print('\nACCIÓN:')
df_viento.loc[df_viento['velocidad_viento_kmh'] > lim_sup, 'velocidad_viento_kmh'] = np.nan
print(f'✓ {(df_viento["velocidad_viento_kmh"].isna().sum())} valores marcados como NaN')
print(f'✓ Luego interpolaremos')

## Paso 5: Visualizar antes/después

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ANTES
ax = axes[0]
normal = df_viento['velocidad_viento_kmh'] <= lim_sup
extremo = df_viento['velocidad_viento_kmh'] > lim_sup
ax.scatter(df_viento[normal].index, df_viento[normal]['velocidad_viento_kmh'], 
          color='blue', s=5, alpha=0.5, label='Normal')
ax.scatter(df_viento[extremo].index, df_viento[extremo]['velocidad_viento_kmh'],
          color='red', s=100, marker='x', linewidth=2, label='Atípicos (400+ m/s)')
ax.axhline(lim_sup, color='red', linestyle='--', alpha=0.5, label=f'Límite IQR: {lim_sup:.0f}')
ax.set_title('ANTES: Viento con 12 valores extremos (ERROR)', fontweight='bold', color='darkred')
ax.set_ylabel('Velocidad (km/h)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 1500)

# DESPUÉS
ax = axes[1]
ax.scatter(df_viento[df_viento['velocidad_viento_kmh'].notna()].index,
          df_viento[df_viento['velocidad_viento_kmh'].notna()]['velocidad_viento_kmh'],
          color='green', s=5, alpha=0.5, label='Normal')
ax.scatter(df_viento[df_viento['velocidad_viento_kmh'].isna()].index,
          [0]*df_viento['velocidad_viento_kmh'].isna().sum(),
          color='orange', s=100, marker='o', edgecolors='black', linewidth=1, label='Convertidos a NaN')
ax.set_title('DESPUÉS: Errores marcados como NaN (para interpolar)', fontweight='bold', color='darkgreen')
ax.set_ylabel('Velocidad (km/h)')
ax.set_xlabel('Índice')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 40)

plt.tight_layout()
plt.show()

## CONCLUSIÓN CASO B

✅ **RESPUESTA: ES ERROR (no evento real)**

**¿Cómo lo confirmamos?**
1. **IQR detecta atípicos**: 400 m/s está 20× fuera del límite
2. **Magnitud imposible**: > registro histórico mundial
3. **Frecuencia anormal**: 0.024% vs. esperado ~0.5%
4. **Aislado en una columna**: Temperatura/presión normales
5. **Patrón de error**: Múltiplos de valores normales (~×100)

**Acción:**
```python
# Marcar como NaN
df.loc[df['viento'] > límite_iqr, 'viento'] = np.nan
# Luego interpolar
df['viento'] = df['viento'].interpolate(method='linear')
```

---
## RESUMEN FINAL

| Aspecto | Caso A (Irradiancia) | Caso B (Viento) |
|---------|----------------------|------------------|
| **Tipo de problema** | Faltantes (NaN) | Valores extremos |
| **%** | 3% (bajo) | 0.024% (muy raro) |
| **Distribución** | Aleatorio en tiempo | Aislado |
| **Decisión** | Interpolar | Marcar como NaN |
| **Razón** | Serie de tiempo | Error imposible |
| **Confirmación** | Visualizar curva | IQR + contexto |

**Lección clave:** Siempre **visualizar y verificar contexto** antes de decidir cómo limpiar.